# Build USL-Suspilne Dataset

End-to-end pipeline: Firebase → annotations → download videos → split → crop signer.

**Output:** `data/usl-suspilne/` with `annotations.csv`, `{train,dev,test}.csv`, and `features/{videoId}/{clip}.mp4`

**Requirements:** `pip install firebase-admin yt-dlp num2words pymorphy3 pymorphy3-dicts-uk`, `ffmpeg` installed, `serviceAccountKey.json` in repo root.

In [ ]:
import sys
from pathlib import Path

ROOT = Path(".").resolve().parent
sys.path.insert(0, str(ROOT / "scripts"))

# All paths in one place
FIREBASE_DIR = ROOT / "data/firebase"
ANNOTATIONS_CSV = ROOT / "data/usl-suspilne/annotations.csv"
SPLITS_CSV = ROOT / "data/cache/splits.csv"
RAW_VIDEOS_DIR = ROOT / "data/cache/raw_videos"
UNCROPPED_DIR = ROOT / "data/cache/uncropped"
FEATURES_DIR = ROOT / "data/usl-suspilne/features"

print(f"Project root: {ROOT}")

## 1. Download Firebase Export

In [2]:
from download_firebase import download_and_save

export_path = download_and_save(output_dir=FIREBASE_DIR)
print(f"\nExport saved to: {export_path}")

Saved to /Users/xandro/code/diploma/ucu-text-to-sign/data/firebase/2026-04-14.json
Updated /Users/xandro/code/diploma/ucu-text-to-sign/data/firebase/latest.json -> 2026-04-14.json
  Videos: 26
  Video captions entries: 26
  Total captions: 6801
  Aligned captions: 5186

Export saved to: /Users/xandro/code/diploma/ucu-text-to-sign/data/firebase/2026-04-14.json


## 2. Build Annotations

Reads the Firebase export (data/firebase/latest.json), finds all captions marked as `aligned`, and writes:
- `data/usl-suspilne/annotations.csv` — `name|text|text_norm|annotator` (for training)
- `data/cache/splits.csv` — `name|video|start|end` (for download/split)

`text_norm` is the normalized text: lowercased, punctuation stripped, numbers converted to words.

Filters out clips shorter than 0.3s and any excluded video IDs.

In [3]:
from build_annotations_from_firebase import build_annotations

result = build_annotations(
    export_path=FIREBASE_DIR / "latest.json",
    output_csv=ANNOTATIONS_CSV,
    splits_csv=SPLITS_CSV,
)
print(f"\n{result['rows']} annotations from {len(result['videos'])} videos")

Videos: 13 complete, 13 incomplete
  train: 1906 clips -> /Users/xandro/code/diploma/ucu-text-to-sign/data/usl-suspilne/train.csv
  dev: 95 clips -> /Users/xandro/code/diploma/ucu-text-to-sign/data/usl-suspilne/dev.csv
  test: 221 clips -> /Users/xandro/code/diploma/ucu-text-to-sign/data/usl-suspilne/test.csv
Wrote 2222 annotations from 13 videos to /Users/xandro/code/diploma/ucu-text-to-sign/data/usl-suspilne/annotations.csv
Skipped 13 incomplete videos (use --all to include)
  2Nnz697BVTw: 85 clips
  A2hCZVvtUSE: 122 clips
  IOflFDS2biE: 484 clips
  K9ouFMtz-s8: 57 clips
  KUDt_SKkPUE: 146 clips
  MSqpwfErl34: 116 clips
  Nyykyn4FpNo: 214 clips
  Q3yRVXmZdGQ: 142 clips
  SG9xYYOLBNI: 38 clips
  cNT6ajjEwVU: 174 clips
  jj5jiyl2mh0: 423 clips
  uGMgleLkjho: 150 clips
  w_LdfLKP_0o: 71 clips

2222 annotations from 13 videos


## 3. Download Videos

Downloads YouTube videos at best available quality.
Skips already-downloaded videos. Use `--force VIDEO_ID` to re-download.

In [4]:
from download_videos import download_all

dl_result = download_all(splits_csv=SPLITS_CSV, dst=RAW_VIDEOS_DIR)

if dl_result["failed"]:
    print(f"\nFailed videos will be excluded in the verify step.")

Found 13 videos in /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/splits.csv

[2Nnz697BVTw]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/2Nnz697BVTw.mp4 already exists
[A2hCZVvtUSE]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/A2hCZVvtUSE.mp4 already exists
[IOflFDS2biE]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/IOflFDS2biE.mp4 already exists
[K9ouFMtz-s8]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/K9ouFMtz-s8.mp4 already exists
[KUDt_SKkPUE]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/KUDt_SKkPUE.mp4 already exists
[MSqpwfErl34]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/MSqpwfErl34.mp4 already exists
[Nyykyn4FpNo]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/Nyykyn4FpNo.mp4 already exists
[Q3yRVXmZdGQ]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/ca

## 4. Split into Clips

Splits videos into sentence-level clips based on timestamps.
Skips already-split clips. Use `--force VIDEO_ID` to re-split.

In [5]:
from split_videos import split_all

split_result = split_all(splits_csv=SPLITS_CSV, raw_dir=RAW_VIDEOS_DIR, dst=UNCROPPED_DIR)

Found 2222 annotations across 13 videos

[2Nnz697BVTw] 85 clips
[A2hCZVvtUSE] 122 clips
[IOflFDS2biE] 484 clips
[K9ouFMtz-s8] 57 clips
[KUDt_SKkPUE] 146 clips
[MSqpwfErl34] 116 clips
[Nyykyn4FpNo] 214 clips
[Q3yRVXmZdGQ] 142 clips
[SG9xYYOLBNI] 38 clips
[cNT6ajjEwVU] 174 clips
[jj5jiyl2mh0] 423 clips
[uGMgleLkjho] 150 clips
[w_LdfLKP_0o] 71 clips

Done. 2222 clips, 0 failed, 0 videos skipped.


## 5. Crop to Signer Region

Crops the bottom-right 510x510 region where the sign language interpreter appears.

In [6]:
from crop_signer import crop_all

crop_result = crop_all(src=UNCROPPED_DIR, dst=FEATURES_DIR)

Found 2222 clips in /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/uncropped

Done. 2222 cropped, 0 failed.


## 6. Verify Dataset

In [7]:
import csv
import subprocess
import json

dataset_dir = ROOT / "data/usl-suspilne"

# Count annotations
with open(ANNOTATIONS_CSV) as f:
    annotations = list(csv.DictReader(f, delimiter="|"))
print(f"Annotations: {len(annotations)}")

# Count clips per video
for vid_dir in sorted(FEATURES_DIR.iterdir()):
    if vid_dir.is_dir():
        clips = list(vid_dir.glob("*.mp4"))
        print(f"  {vid_dir.name}: {len(clips)} clips")

total_clips = len(list(FEATURES_DIR.glob("*/*.mp4")))
print(f"\nTotal clips: {total_clips}")

# Check for missing clips and filter annotations
missing = []
valid = []
for row in annotations:
    clip = FEATURES_DIR / row["name"].split("/")[0] / (row["name"].split("/")[1] + ".mp4")
    if clip.exists():
        valid.append(row)
    else:
        missing.append(row["name"])

if missing:
    print(f"\nWARNING: {len(missing)} annotations without clips — removing from annotations.csv")
    missing_vids = {}
    for name in missing:
        vid = name.split("/")[0]
        missing_vids[vid] = missing_vids.get(vid, 0) + 1
    for vid, count in sorted(missing_vids.items()):
        print(f"  {vid}: {count} missing")

    fieldnames = list(valid[0].keys())
    with open(ANNOTATIONS_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="|")
        writer.writeheader()
        writer.writerows(valid)
    print(f"\nFiltered annotations: {len(valid)} (removed {len(missing)})")
else:
    print("\nAll annotations have matching clips.")

Annotations: 2222
  2Nnz697BVTw: 85 clips
  A2hCZVvtUSE: 122 clips
  IOflFDS2biE: 484 clips
  K9ouFMtz-s8: 57 clips
  KUDt_SKkPUE: 146 clips
  MSqpwfErl34: 116 clips
  Nyykyn4FpNo: 214 clips
  Q3yRVXmZdGQ: 142 clips
  SG9xYYOLBNI: 38 clips
  cNT6ajjEwVU: 174 clips
  jj5jiyl2mh0: 423 clips
  uGMgleLkjho: 150 clips
  w_LdfLKP_0o: 71 clips

Total clips: 2222

All annotations have matching clips.


In [8]:
# Total duration
total_dur = 0
per_video = {}
for clip in sorted(FEATURES_DIR.glob("*/*.mp4")):
    result = subprocess.run(
        ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", str(clip)],
        capture_output=True, text=True
    )
    dur = float(json.loads(result.stdout)["format"]["duration"])
    total_dur += dur
    vid = clip.parent.name
    per_video[vid] = per_video.get(vid, 0) + dur

for vid in sorted(per_video):
    m, s = divmod(per_video[vid], 60)
    print(f"  {vid}: {int(m)}m {s:.0f}s")

h, remainder = divmod(total_dur, 3600)
m, s = divmod(remainder, 60)
print(f"\nTotal duration: {int(h)}h {int(m)}m {s:.0f}s")

  2Nnz697BVTw: 12m 50s
  A2hCZVvtUSE: 15m 32s
  IOflFDS2biE: 34m 28s
  K9ouFMtz-s8: 7m 17s
  KUDt_SKkPUE: 16m 27s
  MSqpwfErl34: 10m 38s
  Nyykyn4FpNo: 19m 17s
  Q3yRVXmZdGQ: 11m 22s
  SG9xYYOLBNI: 20m 45s
  cNT6ajjEwVU: 26m 15s
  jj5jiyl2mh0: 37m 6s
  uGMgleLkjho: 13m 16s
  w_LdfLKP_0o: 9m 42s

Total duration: 3h 54m 55s
